In [1]:
%load_ext autoreload
%reload_ext autoreload

%autoreload 2
%matplotlib inline
import tensorflow
from tensorflow.compat.v1.keras.backend import get_session
tensorflow.compat.v1.disable_v2_behavior()
import math
import kerasAC 
from scipy.special import softmax,expit
from kerasAC.interpret.deepshap import * 
from kerasAC.interpret.profile_shap import * 
from kerasAC.vis import * 
from kerasAC.helpers.transform_bpnet_io import * 
from kerasAC.util import * 
import pandas as pd

## for plotting 

import matplotlib 
from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"]=10,5
plt.rcParams['axes.xmargin'] = 0

font = {'family' : 'normal',
        'weight' : 'bold',
        'size'   : 10}

matplotlib.rc('font', **font)

Instructions for updating:
non-resource variables are not supported in the long term


Using TensorFlow backend.
/users/annashch/kerasAC/kerasAC/vis/plot_letters.py:172: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  min_coords = np.vstack(data.min(0) for data in polygons_data).min(0)
/users/annashch/kerasAC/kerasAC/vis/plot_letters.py:173: FutureWarning: arrays to stack must be passed as a "sequence" type such as list or tuple. Support for non-sequence iterables such as generators is deprecated as of NumPy 1.16 and will raise an error in the future.
  max_coords = np.vstack(data.max(0) for data in polygons_data).max(0)


In [2]:
import pysam 
ref=pysam.FastaFile("/mnt/data/GRCh38_no_alt_analysis_set_GCA_000001405.15.fasta")

In [3]:
#load the model! 
from keras.models import load_model
from keras.utils.generic_utils import get_custom_objects
from kerasAC.metrics import * 
from kerasAC.custom_losses import * 
from keras_genomics.layers.convolutional import RevCompConv1D
import tensorflow_probability as tfp

custom_objects={"recall":recall,
                "sensitivity":recall,
                "specificity":specificity,
                "fpr":fpr,
                "fnr":fnr,
                "precision":precision,
                "f1":f1,
                "ambig_binary_crossentropy":ambig_binary_crossentropy,
                "ambig_mean_absolute_error":ambig_mean_absolute_error,
                "ambig_mean_squared_error":ambig_mean_squared_error,
                "MultichannelMultinomialNLL":MultichannelMultinomialNLL,
                "MultichannelMultinomialMSE":MultichannelMultinomialMSE,                
                "RevCompConv1D": RevCompConv1D}
get_custom_objects().update(custom_objects)
dnase_model=load_model("gm12878.dnase.with.bias.unplugged.0.hdf5")
tf_model=load_model('Spi1ProfileModel_rc.h5')
histone_model=load_model("patience_6_5K_out_9_gm12878.h3k27ac.seed.2345.cs.25.filters.300.naive.range.4.6.to.11.5.0.hdf5")

Instructions for updating:
If using Keras pass *_constraint arguments to layers.
[1] 1
Tensor("loss/profile_predictions_loss/MultichannelMultinomialNLL/truediv:0", shape=(), dtype=float32)
[1, 1] 2
Tensor("loss_1/task0_profile_loss/MultichannelMultinomialNLL/truediv:0", shape=(), dtype=float32)
Tensor("loss_1/task0_profile_loss/MultichannelMultinomialNLL/truediv_1:0", shape=(), dtype=float32)
[1.0, 1.0] 2
[439.7, 439.7] 2
Tensor("loss_2/profile_predictions_loss/MultichannelMultinomialNLL/truediv:0", shape=(), dtype=float32)
Tensor("loss_2/profile_predictions_loss/MultichannelMultinomialNLL/truediv_1:0", shape=(), dtype=float32)
Tensor("loss_2/logcount_predictions_loss/MultichannelMultinomialMSE/mean_squared_error/weighted_loss/value:0", shape=(), dtype=float32) 2
Tensor("loss_2/logcount_predictions_loss/MultichannelMultinomialMSE/mean_squared_error_1/weighted_loss/value:0", shape=(), dtype=float32) 2


In [4]:
import shap
from deeplift.dinuc_shuffle import dinuc_shuffle

def combine_mult_and_diffref(mult, orig_inp, bg_data):
    to_return = []
    for l in [0]:
        projected_hypothetical_contribs = np.zeros_like(bg_data[l]).astype("float")
        assert len(orig_inp[l].shape)==2
        for i in range(orig_inp[l].shape[-1]):
            hypothetical_input = np.zeros_like(orig_inp[l]).astype("float")
            hypothetical_input[:,i] = 1.0
            hypothetical_difference_from_reference = (hypothetical_input[None,:,:]-bg_data[l])
            hypothetical_contribs = hypothetical_difference_from_reference*mult[l]
            projected_hypothetical_contribs[:,:,i] = np.sum(hypothetical_contribs,axis=-1) 
        to_return.append(np.mean(projected_hypothetical_contribs,axis=0))
    to_return.append(np.zeros_like(orig_inp[1]))
    return to_return

def shuffle_several_times(s):
    numshuffles=20
    return [np.array([dinuc_shuffle(s[0]) for i in range(numshuffles)]),
            np.array([s[1] for i in range(numshuffles)])]

tf_counts_explainer = shap.explainers.deep.TFDeepExplainer(
    ([tf_model.input[0], tf_model.input[1]],
     tf.reduce_sum(tf_model.outputs[0],axis=-1)),
    shuffle_several_times,
    combine_mult_and_diffref=combine_mult_and_diffref)

meannormed_logits = (
    tf_model.outputs[1]-
    tf.reduce_mean(tf_model.outputs[1],axis=1)[:,None,:])
stopgrad_meannormed_logits = tf.stop_gradient(meannormed_logits)
softmax_out = tf.nn.softmax(stopgrad_meannormed_logits,axis=1)
weightedsum_meannormed_logits = tf.reduce_sum(softmax_out*meannormed_logits,
                                              axis=(1,2))
tf_profile_explainer = shap.explainers.deep.TFDeepExplainer(
    ([tf_model.input[0], tf_model.input[2]],
     weightedsum_meannormed_logits),
    shuffle_several_times,
    combine_mult_and_diffref=combine_mult_and_diffref)

In [5]:
#Histone chip 
histone_wrapper=([histone_model.input[0],histone_model.input[2]],histone_model.outputs[1][:,0:1])
histone_count_explainer=shap.DeepExplainer(histone_wrapper,
                                     data=create_background_counts_chip,
                                     combine_mult_and_diffref=combine_mult_and_diffref_chip)
histone_prof_explainer=create_explainer(histone_model,ischip=True,task_index=0)



In [6]:
#DNASE
dnase_wrapper=(dnase_model.input,dnase_model.outputs[1][:,0:1])
dnase_count_explainer=shap.DeepExplainer(dnase_wrapper,
                                        data=create_background_atac,
                                        combine_mult_and_diffref=combine_mult_and_diffref_atac)
dnase_prof_explainer=create_explainer(dnase_model,ischip=False,task_index=0)


In [7]:
## source signal tracks 
import pyBigWig
#DNASE 
dnase_bw=pyBigWig.open("/oak/stanford/groups/akundaje/projects/atlas/dnase_processed/dnase/13da5ebe-0941-4855-8599-40bbcc5c58b4/call-bowtie2/shard-0/execution/ENCSR000EMT.merged.bam.bpnet.unstranded.bw",'r')

#SPI1 TF 
tf_pos=pyBigWig.open("/srv/scratch/annashch/chrombpnet/gm12878_tf_dnase_histone/spi1_tf/pos_strand.bw")
tf_neg=pyBigWig.open("/srv/scratch/annashch/chrombpnet/gm12878_tf_dnase_histone/spi1_tf/neg_strand.bw")
tf_control_pos=pyBigWig.open("/srv/scratch/annashch/chrombpnet/gm12878_tf_dnase_histone/spi1_tf/control_neg_strand.bw")
tf_control_neg=pyBigWig.open("/srv/scratch/annashch/chrombpnet/gm12878_tf_dnase_histone/spi1_tf/control_pos_strand.bw")

#H3K27AC 
chip_pos=pyBigWig.open("/srv/scratch/annashch/chrombpnet/gm12878_tf_dnase_histone/h3k27ac/K562.merged.ENCSR000APK.gz.bam.bpnet.plus.bw")
chip_neg=pyBigWig.open("/srv/scratch/annashch/chrombpnet/gm12878_tf_dnase_histone/h3k27ac/K562.merged.ENCSR000APK.gz.bam.bpnet.minus.bw")
chip_control_pos=pyBigWig.open("/srv/scratch/annashch/chrombpnet/gm12878_tf_dnase_histone/h3k27ac/K562.merged.control.ENCSR000AKY.gz.bam.bpnet.plus.bw")
chip_control_neg=pyBigWig.open("/srv/scratch/annashch/chrombpnet/gm12878_tf_dnase_histone/h3k27ac/K562.merged.control.ENCSR000AKY.gz.bam.bpnet.minus.bw")


In [15]:
#USE HG38 
regions=pd.read_csv("bQTL.for.ism.csv",header=0,sep='\t')
#regions=pd.read_csv("bQTL.intersection.csv",header=0,sep='\t')
regions

,Chrom,hg19pos0,hg38pos0,Major,Minor,Group,Rsid
0,chr14,22590585,22122627,T,A,StrongbQTL,rs11157420
1,chr22,44116809,43720929,C,G,StrongbQTL,rs5764238
2,chr4,37814568,37812946,C,T,StrongbQTL,rs116009584
3,chr7,157313211,157520517,A,G,StrongbQTL,rs4716753
4,chr8,25242971,25385455,T,C,StrongbQTL,rs925032
5,chr1,64029605,63563934,G,A,StrongbQTL,rs217474
6,chr7,100230863,100633240,G,C,StrongbQTL,rs34242818
7,chr1,144534082,149162899,T,C,StrongbQTL,rs1184471061
8,chr6,26660654,26660426,G,C,StrongbQTL,rs2451713
9,chr15,52536308,52244111,G,A,Av,rs4774616


In [16]:
def get_preds(model,inputs):
    preds=model.predict(inputs)
    prof=np.squeeze(preds[0])
    probs=softmax(prof,axis=0)
    count=np.squeeze(preds[1])  
    count_track=probs*np.exp(count) 
    return prof,count,probs,count_track

In [17]:
def analyze_dnase(ref,
            chrom,
            summit,
            ref_allele,
            alt_allele,
            rsid,
            model,
            count_explainer,
            prof_explainer,
            flank=673,
            output_bp=1000,
            bigwig_unstranded=None):
    #get the reference and alternate one-hot-encoded sequences 
    seq=ref.fetch(chrom,summit-flank,summit+flank)
    
    ref_seq=seq[0:flank]+ref_allele+seq[flank+1::]
    assert len(ref_seq)==2*flank
    ref_onehot=one_hot_encode([ref_seq])
    
    alt_seq=seq[0:flank]+alt_allele+seq[flank+1::]
    assert len(alt_seq)==2*flank
    alt_onehot=one_hot_encode([alt_seq])
    
    #get the bigwig labels 
    labels_unstranded=np.nan_to_num(bigwig_unstranded.values(chrom,summit-flank,summit+flank))
    model_input_ref=ref_onehot
    model_input_alt=alt_onehot
    #get predictions for reference & alternate allele 
    prof_ref,count_ref,probs_ref,count_track_ref=get_preds(model,model_input_ref)
    prof_alt,count_alt,probs_alt,count_track_alt=get_preds(model,model_input_alt)

        
    #get the log odds blast radius track 
    blast_radius_track=np.log(probs_ref)-np.log(probs_alt)
    
    #get deepSHAP scores for ref & alt alleles     
    profile_explanations_ref=prof_explainer(ref_onehot,None)*ref_onehot
    count_explanations_ref=np.squeeze(count_explainer.shap_values(ref_onehot)[0])*ref_onehot 

    profile_explanations_alt=prof_explainer(alt_onehot,None)*alt_onehot
    count_explanations_alt=np.squeeze(count_explainer.shap_values(alt_onehot)[0])*alt_onehot  
    
    #look at dimensions of size 1000 
    offset=int((2*flank-output_bp)/2)
    labels_unstranded=labels_unstranded[offset:offset+output_bp]
    profile_explanations_ref=np.squeeze(profile_explanations_ref)[offset:offset+output_bp,:]
    profile_explanations_alt=np.squeeze(profile_explanations_alt)[offset:offset+output_bp,:]
    count_explanations_ref=np.squeeze(count_explanations_ref)[offset:offset+output_bp,:]
    count_explanations_alt=np.squeeze(count_explanations_alt)[offset:offset+output_bp,:]
    
    
    return labels_unstranded,count_track_ref,count_track_alt,blast_radius_track,profile_explanations_ref,count_explanations_ref, profile_explanations_alt, count_explanations_alt

In [18]:
def analyze_histone(ref,
            chrom,
            summit,
            ref_allele,
            alt_allele,
            rsid,
            model,
            count_explainer,
            prof_explainer,
            bigwig_plus,
            bigwig_minus,
            control_plus,
            control_minus,
            flank=4000,
            output_bp=1000):
    #get the reference and alternate one-hot-encoded sequences 
    seq=ref.fetch(chrom,summit-flank,summit+flank)
    
    ref_seq=seq[0:flank]+ref_allele+seq[flank+1::]
    assert len(ref_seq)==2*flank
    ref_onehot=one_hot_encode([ref_seq])
    
    alt_seq=seq[0:flank]+alt_allele+seq[flank+1::]
    assert len(alt_seq)==2*flank
    alt_onehot=one_hot_encode([alt_seq])
    
    #stranded inputs mean we are using chipseq
    labels_plus=np.nan_to_num(bigwig_plus.values(chrom,summit-flank,summit+flank))
    labels_minus=np.nan_to_num(bigwig_minus.values(chrom,summit-flank,summit+flank))
    control_labels_plus=np.expand_dims(np.nan_to_num(control_plus.values(chrom,summit-2500,summit+2500)),axis=1)
    control_labels_minus=np.expand_dims(np.nan_to_num(control_minus.values(chrom,summit-2500,summit+2500)),axis=1)
    control_input_profile=np.expand_dims(np.concatenate((control_labels_plus,control_labels_minus),axis=-1),axis=0)
    #print("histone_control_input_profile"+str(control_input_profile.shape))
    control_count_plus=np.expand_dims(np.expand_dims(np.array(np.log(np.sum(control_labels_plus))),axis=0),axis=1)
    control_count_minus=np.expand_dims(np.expand_dims(np.array(np.log(np.sum(control_labels_minus))),axis=0),axis=1)    
    control_input_count=np.concatenate((control_count_plus,control_count_minus),axis=-1)
    #print("histone_control_input_count:"+str(control_input_count.shape))
    
    model_input_ref=[ref_onehot, control_input_profile, control_input_count]
    model_input_alt=[alt_onehot, control_input_profile, control_input_count]
        
    #get predictions for reference & alternate allele 
    prof_ref,count_ref,probs_ref,count_track_ref=get_preds(model,model_input_ref)
    prof_alt,count_alt,probs_alt,count_track_alt=get_preds(model,model_input_alt)

    #get the log odds blast radius track 
    blast_radius_track_plus=np.log(probs_ref[:,0])-np.log(probs_alt[:,0])
    blast_radius_track_minus=np.log(probs_ref[:,1])-np.log(probs_alt[:,1])

    #get deepSHAP scores for ref & alt alleles 
    count_explanations_ref=count_explainer.shap_values([ref_onehot,control_input_count])[0][0]*ref_onehot
    profile_explanations_ref=prof_explainer(ref_onehot,control_input_profile)[0]*ref_onehot
    
    count_explanations_alt=count_explainer.shap_values([alt_onehot,control_input_count])[0][0]*alt_onehot
    profile_explanations_alt=prof_explainer(alt_onehot,control_input_profile)[0]*alt_onehot
    
        
    #look at dimensions of size 1000 
    labels_plus=labels_plus[3500:4500]
    labels_minus=labels_minus[3500:4500]
    profile_explanations_ref=np.squeeze(profile_explanations_ref)[3500:4500,:]
    profile_explanations_alt=np.squeeze(profile_explanations_alt)[3500:4500,:]

    count_explanations_ref=np.squeeze(count_explanations_ref)[3500:4500,:]
    count_explanations_alt=np.squeeze(count_explanations_alt)[3500:4500,:]
    
    count_track_ref=count_track_ref[2000:3000,:]
    count_track_alt=count_track_alt[2000:3000,:]
    
    blast_radius_track_plus=blast_radius_track_plus[2000:3000]
    blast_radius_track_minus=blast_radius_track_minus[2000:3000]
    #return outputs 
    return labels_plus,labels_minus,count_track_ref,count_track_alt,blast_radius_track_plus,blast_radius_track_minus,profile_explanations_ref,count_explanations_ref, profile_explanations_alt, count_explanations_alt
 

In [19]:
def rolling_window(a, window):
    shape = a.shape[:-1] + (a.shape[-1] - window + 1, window)
    strides = a.strides + (a.strides[-1],)
    return np.lib.stride_tricks.as_strided(a, shape=shape, strides=strides)

def smooth_profiles(profiles, smoothing_window):
    assert len(profiles.shape)==3
    leftpadlen = int((smoothing_window-1)/2)
    rightpadlen =\
        (smoothing_window-1)-int((smoothing_window-1)/2)
    padded_profiles = np.pad(
        array=profiles,
        pad_width=((0,0),(leftpadlen, rightpadlen), (0,0)),
        mode='edge')
    smoothed_profiles = np.mean(rolling_window(
                        a=padded_profiles.transpose(0,2,1),
                        window=smoothing_window), axis=-1).transpose((0,2,1))
    return smoothed_profiles

def get_preds_tf(model,inputs):
    preds=model.predict(inputs)
    prof=np.squeeze(preds[1])
    probs=softmax(prof,axis=0)
    count=np.squeeze(preds[0])  
    count_track=probs*np.exp(count)
    return prof,count,probs,count_track

def analyze_tf(ref,
            chrom,
            summit,
            ref_allele,
            alt_allele,
            rsid,
            model,
            count_explainer,
            prof_explainer,
            bigwig_plus,
            bigwig_minus,
            control_plus,
            control_minus,
            flank=673,
            output_bp=1000):
    #get the reference and alternate one-hot-encoded sequences 
    seq=ref.fetch(chrom,summit-flank,summit+flank)
    
    ref_seq=seq[0:flank]+ref_allele+seq[flank+1::]
    assert len(ref_seq)==2*flank
    ref_onehot=one_hot_encode([ref_seq])
    
    alt_seq=seq[0:flank]+alt_allele+seq[flank+1::]
    assert len(alt_seq)==2*flank
    alt_onehot=one_hot_encode([alt_seq])
    
    #stranded inputs mean we are using chipseq
    labels_plus=np.nan_to_num(bigwig_plus.values(chrom,summit-flank,summit+flank))
    labels_minus=np.nan_to_num(bigwig_minus.values(chrom,summit-flank,summit+flank))
    control_labels_plus=np.nan_to_num(control_plus.values(chrom,summit-int(output_bp/2),summit+int(output_bp/2)))
    control_labels_minus=np.nan_to_num(control_minus.values(chrom,summit-int(output_bp/2),summit+int(output_bp/2)))
    control_labels=np.expand_dims(np.expand_dims(control_labels_plus+control_labels_minus,axis=0),axis=2)
    control_input_profile=np.concatenate((control_labels,smooth_profiles(profiles=control_labels,smoothing_window=50)),axis=2)    
    control_input_count=np.expand_dims(np.expand_dims(np.array(np.log(np.sum(control_labels_plus)+np.sum(control_labels_minus)+1)),axis=0),axis=1)
    model_input_ref=[ref_onehot, control_input_count, control_input_profile]
    model_input_alt=[alt_onehot, control_input_count, control_input_profile]
        
    #get predictions for reference & alternate allele     
    prof_ref,count_ref,probs_ref,count_track_ref=get_preds_tf(model,model_input_ref)
    prof_alt,count_alt,probs_alt,count_track_alt=get_preds_tf(model,model_input_alt)
  
    #get the log odds blast radius track 
    blast_radius_track_plus=np.log(probs_ref[:,0])-np.log(probs_alt[:,0])
    blast_radius_track_minus=np.log(probs_ref[:,1])-np.log(probs_alt[:,1])

    #get deepSHAP scores for ref & alt alleles 
    count_explanations_ref=count_explainer.shap_values([ref_onehot,control_input_count])[0]*ref_onehot
    profile_explanations_ref=prof_explainer.shap_values([ref_onehot,control_input_profile])[0]*ref_onehot
    
    count_explanations_alt=count_explainer.shap_values([alt_onehot,control_input_count])[0]*alt_onehot
    profile_explanations_alt=prof_explainer.shap_values([alt_onehot,control_input_profile])[0]*alt_onehot
    
    #look at dimensions of size 1000 
    offset=int((2*flank-output_bp)/2)
    labels_plus=labels_plus[offset:offset+output_bp]
    labels_minus=labels_minus[offset:offset+output_bp]
    profile_explanations_ref=np.squeeze(profile_explanations_ref)[offset:offset+output_bp,:]
    profile_explanations_alt=np.squeeze(profile_explanations_alt)[offset:offset+output_bp,:]
    count_explanations_ref=np.squeeze(count_explanations_ref)[offset:offset+output_bp,:]
    count_explanations_alt=np.squeeze(count_explanations_alt)[offset:offset+output_bp,:]
        
    #return outputs 
    return labels_plus,labels_minus,count_track_ref,count_track_alt,blast_radius_track_plus,blast_radius_track_minus,profile_explanations_ref,count_explanations_ref, profile_explanations_alt, count_explanations_alt
 
     

In [20]:
from scipy.special import softmax 
def make_plot(dnase_labels_unstranded,
              dnase_count_track_ref,
              dnase_count_track_alt,
              dnase_blast_radius_track,
              dnase_profile_explanations_ref, 
              dnase_count_explanations_ref, 
              dnase_profile_explanations_alt, 
              dnase_count_explanations_alt,
              tf_labels_plus,
              tf_labels_minus,
              tf_count_track_ref,
              tf_count_track_alt,
              tf_blast_radius_track_plus,
              tf_blast_radius_track_minus,
              tf_profile_explanations_ref,
              tf_count_explanations_ref, 
              tf_profile_explanations_alt, 
              tf_count_explanations_alt,
              histone_labels_plus,
              histone_labels_minus, 
              histone_count_track_ref, 
              histone_count_track_alt, 
              histone_blast_radius_track_plus, 
              histone_blast_radius_track_minus, 
              histone_profile_explanations_ref,
              histone_count_explanations_ref, 
              histone_profile_explanations_alt, 
              histone_count_explanations_alt,
              title,
              xmin=0,
              xmax=1000): 
    plt.rcParams["figure.figsize"]=35,25
    f,axes=plt.subplots(7, 3,sharex='row')
    
    #labels count track 
    axes[0,0].plot(dnase_labels_unstranded)  
    axes[0,0].set_title("DNASE labels",fontsize=20)
    axes[0,1].plot(tf_labels_plus)
    axes[0,1].plot(tf_labels_minus*-1)
    axes[0,1].set_title("SPI1 TF Chip-seq labels",fontsize=20)
    axes[0,2].plot(histone_labels_plus)
    axes[0,2].plot(histone_labels_minus*-1)
    axes[0,2].set_title("H3K27ac Chip-seq labels",fontsize=20)
    
    #ref vs alt preds 
    axes[1,0].plot(dnase_count_track_ref,label='ref',color='b')
    axes[1,0].plot(dnase_count_track_alt,label='alt',color='r')    
    axes[1,0].set_title("DNASE Predicted Count Track (Ref vs Alt)",fontsize=20)
    axes[1,0].legend()
    axes[1,1].plot(tf_count_track_ref[:,0],label='ref',color='b')
    axes[1,1].plot(tf_count_track_ref[:,1]*-1,label='ref',color='b')
    axes[1,1].plot(tf_count_track_alt[:,0],label='alt',color='r')
    axes[1,1].plot(tf_count_track_alt[:,1]*-1,label='alt',color='r')
    axes[1,1].set_title("SPI1 TF Predicted Count Track (Ref vs Alt)",fontsize=20)
    axes[1,2].plot(histone_count_track_ref[:,0],label='ref',color='b')
    axes[1,2].plot(histone_count_track_ref[:,1]*-1,label='ref',color='b')
    axes[1,2].plot(histone_count_track_alt[:,0],label='alt',color='r')
    axes[1,2].plot(histone_count_track_alt[:,1]*-1,label='alt',color='r')
    axes[1,2].set_title("H3K27ac Predicted Count Track (Ref vs Alt)",fontsize=20)
  
    
    # Blast radius 
    axes[2,0].plot(dnase_blast_radius_track)
    axes[2,0].set_title("DNASE Blast Radius log(prob(ref))-log(prob(alt))",fontsize=20)
    axes[2,1].plot(tf_blast_radius_track_plus)
    axes[2,1].plot(tf_blast_radius_track_minus)
    axes[2,1].set_title("SPI1 TF Blast Radius",fontsize=20)
    axes[2,2].plot(histone_blast_radius_track_plus)
    axes[2,2].plot(histone_blast_radius_track_minus)
    axes[2,2].set_title("H3K27ac Blast Radius",fontsize=20)
    
    #DeepSHAP Profile Ref 
    axes[3,0]=plot_bases_on_ax(dnase_profile_explanations_ref,axes[3,0],show_ticks=False)
    axes[3,0].set_title("DNASE Ref Prof SHAP",fontsize=20)
    axes[3,1]=plot_bases_on_ax(tf_profile_explanations_ref,axes[3,1],show_ticks=False)
    axes[3,1].set_title("TF Ref Prof SHAP",fontsize=20)
    axes[3,2]=plot_bases_on_ax(histone_profile_explanations_ref,axes[3,2],show_ticks=False)
    axes[3,2].set_title("H3K27ac Ref Prof SHAP",fontsize=20)
    
    #DeepSHAP Profile Alt 
    axes[4,0]=plot_bases_on_ax(dnase_profile_explanations_alt,axes[4,0],show_ticks=False)
    axes[4,0].set_title("DNASE Alt Prof SHAP",fontsize=20)
    axes[4,1]=plot_bases_on_ax(tf_profile_explanations_alt,axes[4,1],show_ticks=False)
    axes[4,1].set_title("TF Alt Prof SHAP",fontsize=20)
    axes[4,2]=plot_bases_on_ax(histone_profile_explanations_alt,axes[4,2],show_ticks=False)
    axes[4,2].set_title("H3K27ac Alt Prof SHAP",fontsize=20)  
        
    #DeepSHAP Count Ref
    axes[5,0]=plot_bases_on_ax(dnase_count_explanations_ref,axes[5,0],show_ticks=False)
    axes[5,0].set_title("DNASE Ref Count SHAP",fontsize=20)
    axes[5,1]=plot_bases_on_ax(tf_count_explanations_ref,axes[5,1],show_ticks=False)
    axes[5,1].set_title("TF Ref Count SHAP",fontsize=20)
    axes[5,2]=plot_bases_on_ax(histone_count_explanations_ref,axes[5,2],show_ticks=False)
    axes[5,2].set_title("H3K27ac Ref Count SHAP",fontsize=20)  
    
    #DeepSHAP Count Alt 
    axes[6,0]=plot_bases_on_ax(dnase_count_explanations_alt,axes[6,0],show_ticks=False)
    axes[6,0].set_title("DNASE Alt Count SHAP",fontsize=20)
    axes[6,1]=plot_bases_on_ax(tf_count_explanations_alt,axes[6,1],show_ticks=False)
    axes[6,1].set_title("TF Alt Count SHAP",fontsize=20)
    axes[6,2]=plot_bases_on_ax(histone_count_explanations_alt,axes[6,2],show_ticks=False)
    axes[6,2].set_title("H3K27ac Alt Count SHAP",fontsize=20)  
    
    for i in range(7):
        for j in range(3):
            axes[i,j].set_xlim(xmin,xmax)
            axes[i,j].set_xticks(list(range(xmin, xmax, 50)))  
    
    #set ylim for deepSHAP 
    dnase_shap_min=min([np.min(dnase_profile_explanations_ref),
           np.min(dnase_profile_explanations_alt),
           np.min(dnase_count_explanations_ref),
           np.min(dnase_count_explanations_alt)])
    dnase_shap_max=max([np.max(dnase_profile_explanations_ref),
           np.max(dnase_profile_explanations_alt),
           np.max(dnase_count_explanations_ref),
           np.max(dnase_count_explanations_alt)])
    axes[3,0].set_ylim(dnase_shap_min,dnase_shap_max)
    axes[4,0].set_ylim(dnase_shap_min,dnase_shap_max)
    axes[5,0].set_ylim(dnase_shap_min,dnase_shap_max)
    axes[6,0].set_ylim(dnase_shap_min,dnase_shap_max)

    tf_shap_min=min([np.min(tf_profile_explanations_ref),
           np.min(tf_profile_explanations_alt),
           np.min(tf_count_explanations_ref),
           np.min(tf_count_explanations_alt)])
    tf_shap_max=max([np.max(tf_profile_explanations_ref),
           np.max(tf_profile_explanations_alt),
           np.max(tf_count_explanations_ref),
           np.max(tf_count_explanations_alt)])
    axes[3,1].set_ylim(tf_shap_min,tf_shap_max)
    axes[4,1].set_ylim(tf_shap_min,tf_shap_max)
    axes[5,1].set_ylim(tf_shap_min,tf_shap_max)
    axes[6,1].set_ylim(tf_shap_min,tf_shap_max)
    
    histone_shap_min=min([np.min(histone_profile_explanations_ref),
           np.min(histone_profile_explanations_alt),
           np.min(histone_count_explanations_ref),
           np.min(histone_count_explanations_alt)])
    histone_shap_max=max([np.max(histone_profile_explanations_ref),
           np.max(histone_profile_explanations_alt),
           np.max(histone_count_explanations_ref),
           np.max(histone_count_explanations_alt)])
    axes[3,2].set_ylim(histone_shap_min,histone_shap_max)
    axes[4,2].set_ylim(histone_shap_min,histone_shap_max)
    axes[5,2].set_ylim(histone_shap_min,histone_shap_max)
    axes[6,2].set_ylim(histone_shap_min,histone_shap_max) 
    
    plt.suptitle(title,fontsize=20)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(title.replace('/',':')+'.png',format='png',dpi=300)

 

In [ ]:
import pdb
for index,row in regions.iterrows(): 
    chrom=row['Chrom']
    summit=row['hg38pos0']
    ref_allele=row['Major']
    alt_allele=row['Minor']
    rsid=row['Rsid']
    print(index)
    
    #DNASE 
    dnase_labels_unstranded,dnase_count_track_ref,dnase_count_track_alt,dnase_blast_radius_track,dnase_profile_explanations_ref, dnase_count_explanations_ref, dnase_profile_explanations_alt, dnase_count_explanations_alt=analyze_dnase(ref=ref,
            chrom=chrom,
            summit=summit,
            ref_allele=ref_allele,
            alt_allele=alt_allele,
            rsid=rsid,
            bigwig_unstranded=dnase_bw,
            model=dnase_model,
            count_explainer=dnase_count_explainer,
            prof_explainer=dnase_prof_explainer,
            flank=673)
    
    #TF CHIP SEQ 
    tf_labels_plus,tf_labels_minus,tf_count_track_ref,tf_count_track_alt,tf_blast_radius_track_plus,tf_blast_radius_track_minus,tf_profile_explanations_ref,tf_count_explanations_ref, tf_profile_explanations_alt, tf_count_explanations_alt=analyze_tf(ref=ref,
            chrom=chrom,
            summit=summit,
            ref_allele=ref_allele,
            alt_allele=alt_allele,
            rsid=rsid,
            bigwig_plus=tf_pos,
            bigwig_minus=tf_neg,
            control_plus=tf_control_pos,
            control_minus=tf_control_neg,
            count_explainer=tf_counts_explainer,
            prof_explainer=tf_profile_explainer,
            flank=673,
            model=tf_model)
    
    #HISTONE CHIP SEQ 
    histone_labels_plus,histone_labels_minus, histone_count_track_ref, histone_count_track_alt, histone_blast_radius_track_plus, histone_blast_radius_track_minus, histone_profile_explanations_ref,histone_count_explanations_ref, histone_profile_explanations_alt, histone_count_explanations_alt=analyze_histone(ref=ref,
            chrom=chrom,
            summit=summit,
            ref_allele=ref_allele,
            alt_allele=alt_allele,
            rsid=rsid,
            bigwig_plus=chip_pos,
            bigwig_minus=chip_neg,
            control_plus=chip_control_pos,
            control_minus=chip_control_neg,
            count_explainer=histone_count_explainer,
            prof_explainer=histone_prof_explainer,
            flank=4000,
            model=histone_model)
 
    #make plot 
    make_plot(dnase_labels_unstranded,
              dnase_count_track_ref,
              dnase_count_track_alt,
              dnase_blast_radius_track,
              dnase_profile_explanations_ref, 
              dnase_count_explanations_ref, 
              dnase_profile_explanations_alt, 
              dnase_count_explanations_alt,
              tf_labels_plus,
              tf_labels_minus,
              tf_count_track_ref,
              tf_count_track_alt,
              tf_blast_radius_track_plus,
              tf_blast_radius_track_minus,
              tf_profile_explanations_ref,
              tf_count_explanations_ref, 
              tf_profile_explanations_alt, 
              tf_count_explanations_alt,
              histone_labels_plus,
              histone_labels_minus, 
              histone_count_track_ref, 
              histone_count_track_alt, 
              histone_blast_radius_track_plus, 
              histone_blast_radius_track_minus, 
              histone_profile_explanations_ref,
              histone_count_explanations_ref, 
              histone_profile_explanations_alt, 
              histone_count_explanations_alt,
              rsid+":"+chrom+":"+str(summit)+":ref="+ref_allele+":alt="+alt_allele)

0
(1000,)
(1000,)
173
(1, 1000, 2)
(1, 1)
(1000, 2)
(1000, 2)
(1000, 2)
(1000, 2)
173
histone_control_input_profile(1, 5000, 2)
histone_control_input_count:(1, 2)
(5000, 2)
(5000, 2)
1
(1000,)
(1000,)
173
(1, 1000, 2)
(1, 1)
(1000, 2)
(1000, 2)
(1000, 2)
(1000, 2)
173
histone_control_input_profile(1, 5000, 2)
histone_control_input_count:(1, 2)
(5000, 2)
(5000, 2)
2
(1000,)
(1000,)
173
(1, 1000, 2)
(1, 1)
(1000, 2)
(1000, 2)
(1000, 2)
(1000, 2)
173
histone_control_input_profile(1, 5000, 2)
histone_control_input_count:(1, 2)
(5000, 2)
(5000, 2)
